# Os arquivos e a Copa nova

## Task 3: controlar os arquivos gravados e carregar uma Copa nova sem apagar o histórico

Dois problemas do vídeo 5.2: os arquivos pequenos e a Copa de 2026.

In [ ]:
import sys
sys.path.insert(0, "../../src")

from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession, functions as F

construtor = (
    SparkSession.builder
    .appName("arena-copa-nova")
    .master("local[*]")
    .config("spark.sql.warehouse.dir", "../../data/warehouse")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)
spark = configure_spark_with_delta_pip(construtor).getOrCreate()

RAW = "../../data/raw"
RAW_2026 = "../../data/raw_2026"
OUT = "../../data/out"

spark

A célula abaixo já vem pronta: o código dos vídeos 5.1 e 5.2 (histórico e `contar_arquivos`).

In [ ]:
# vídeo 5.1: as partidas do projeto
from wc_pipeline.schemas import MATCHES
from wc_pipeline.transformacoes import limpar_partidas

def ler_partidas(pasta):
    return limpar_partidas(
        spark.read.option("header", "true").schema(MATCHES).csv(f"{pasta}/WorldCupMatches.csv")
    )

historico = ler_partidas(RAW)

# vídeo 5.2: a contagem de arquivos
def contar_arquivos(pasta):
    return len(spark.read.format("delta").load(pasta).inputFiles())

## repartition e coalesce

| | `repartition` | `coalesce` |
|---|---|---|
| O que faz | redistribui em N partições, ou pelo valor de uma coluna | junta partições vizinhas |
| Shuffle | **sim** | **não** |
| Pode aumentar as partições? | sim | não |

Cada partição do `coalesce` continua com todas as Copas. Para **um arquivo por pasta**, as linhas de cada ano precisam
estar na mesma partição: `repartition("ano")`, o `hashpartitioning` do vídeo 4.3.

Um arquivo por pasta, ao custo de um shuffle na escrita.

## Uma Copa nova na saída por Copa

E com o `overwrite` do vídeo 5.1?

O que sobrou? O overwrite **estático** (padrão) troca a tabela inteira. Com `partitionOverwriteMode` em `dynamic`, ele
substitui **só as pastas do lote novo**. Primeiro, o histórico de volta:

Agora 2026, com o overwrite dinâmico:

**Rode a célula de novo**, como se o job tivesse rodado outra vez.

Com o modo dinâmico, rodar outra vez não muda o resultado. Isso é **idempotência**.

| Modo | Rodou duas vezes | Histórico |
|---|---|---|
| `append` | duplicou | preservado |
| `overwrite` (estático) | ok | **fora da versão atual** |
| `overwrite` com `dynamic` | ok | **preservado** |

## A carga de verdade

As decisões estão tomadas. Agora, as **tabelas de verdade**: `data/out/partidas` e `data/out/resumo_selecoes`. A mesma
escrita vale para as duas e para cada lote: uma função.

A próxima célula já vem pronta: ela só apaga as duas tabelas, para a carga começar do zero.

In [ ]:
import shutil
for tabela in ["partidas", "resumo_selecoes"]:
    shutil.rmtree(f"{OUT}/{tabela}", ignore_errors=True)

Primeiro o histórico, depois 2026. O resumo vem das funções da aula 2:

## Resultado da task 3

As tabelas de verdade gravadas, com as 23 Copas.

## Para consultar depois

- **`replaceWhere`:** no Delta, `.option("replaceWhere", "ano = 2026")` substitui só as linhas que atendem à condição,
  um jeito mais explícito de dizer qual parte da tabela o lote troca.
- **`repartition` com número e coluna:** `repartition(4, "ano")` define o número de partições e a coluna ao mesmo
  tempo.
- **Partições depois do `repartition("ano")`:** o número vem de `spark.sql.shuffle.partitions` (200), e o AQE junta as
  pequenas, como no vídeo 4.3.

## Encerrando

Ao terminar o notebook, encerre a sessão Spark. Isso libera a memória e a porta da Spark UI para o próximo notebook.

In [ ]:
spark.stop()